In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import  mean_squared_error
import numpy as np
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression

In [11]:
df = pd.read_csv("breast-cancer.csv")
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [12]:
def pre_process_breast(df):
    df= df.dropna()
    df = df.drop(columns=("id"))
    scaler = StandardScaler()
    X = df.drop('diagnosis',axis=1)
    y = df.diagnosis
    X_scaled = scaler.fit_transform(X)
    return X_scaled, y

In [18]:
scaler = MinMaxScaler()
X, y = pre_process_breast(df)
X_train, X_test,y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

clf = LogisticRegression()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
clf.fit(X_train_s, y_train)

y_pred = clf.predict(X_test_s)

In [19]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[67  0]
 [ 4 43]]
              precision    recall  f1-score   support

           B       0.94      1.00      0.97        67
           M       1.00      0.91      0.96        47

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



Normalização porque generaliza melhor

In [40]:
def pre_process_liver(df):
    df= df.drop('Gender', axis=1)
    df = df.dropna()
    #df['Dataset'] = df['Dataset'].replace(2,0)
    df = df.rename(columns={'Dataset': 'Disease'})
    X = df.drop(columns=('Disease'))
    y = df.Disease
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled , y

In [45]:
df2 = pd.read_csv('indian_patient_records.zip')
X, y = pre_process_liver(df2)
X_train, X_test,y_train, y_test = train_test_split(X, y, test_size=0.2,stratify=y ,random_state=0)
clf = LogisticRegression()
clf.fit(X_train, y_train)
cross = cross_validate(clf,  X, y, cv=10,scoring=["accuracy", "f1"] ,return_train_score=True)
print("Accuracy: ", cross["test_accuracy"].mean())
print("Mean F1-score: ", cross['test_f1'].mean())

Accuracy:  0.7202964307320023
Mean F1-score:  0.8258195690508374


In [46]:
df3 = pd.read_csv('indian_patient_records.zip')
df3 = df3.dropna()
df3 = pd.get_dummies(df3, drop_first=True,columns=["Gender"])
numeric_cols = ['Age', 'Total_Bilirubin', 'Direct_Bilirubin', 'Alkaline_Phosphotase', 'Alamine_Aminotransferase',
                'Aspartate_Aminotransferase', 'Total_Protiens', 'Albumin', 'Albumin_and_Globulin_Ratio']
scaler_standard = StandardScaler()
df3[numeric_cols]= scaler_standard.fit_transform(df3[numeric_cols])
X= df3.drop(columns='Dataset')
y = df3['Dataset']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y ,random_state=0)
log = LogisticRegression(random_state=0)
metrics = cross_validate(log, X, y, cv=10, scoring=["accuracy", "f1_macro"],return_train_score=True)
print("Cross-Validation Results:")
print("Mean Macro F1: ", metrics['test_f1_macro'].mean())
print("Mean Accuracy: ", metrics['test_accuracy'].mean())

Cross-Validation Results:
Mean Macro F1:  0.5458028896629882
Mean Accuracy:  0.7237447065940714


The one-hot encoding model generalizes better